# 🛡️ MODEL 1: Isolation Forest (Unsupervised Anomaly Detection)
### AVISHKAR'26 Hackathon | Team Mavericks (ID: AV26-914D8EC853)
**Objective**: Detect zero-day threats, unlabelled outliers, and structural burst anomalies across Bitcoin transactions without relying on prior labels.

### 1. Install & Import Libraries

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib joblib

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score

os.makedirs('models', exist_ok=True)
print('Libraries imported successfully.')

### 2. Generate / Load Bitcoin Transaction Data
Simulating normal peer-to-peer / exchange baseline transfers alongside novel zero-day attack topologies.

In [ ]:
np.random.seed(42)
records = []

# 4,000 Normal / Licit Transactions (Retail users, exchanges, miners)
for i in range(4000):
    amt = float(np.clip(np.random.lognormal(0.2, 1.1), 0.001, 50.0))
    in_deg = int(np.clip(np.random.poisson(1.8), 1, 6))
    out_deg = int(np.clip(np.random.poisson(2.1), 1, 4))
    vel = float(np.random.exponential(600))
    records.append({
        'txId': f'norm_{i}',
        'in_degree': in_deg,
        'out_degree': out_deg,
        'tx_amount': round(amt, 6),
        'tx_fee': round(amt * np.random.uniform(0.0001, 0.001), 6),
        'hop_velocity': round(vel, 2),
        'fan_out_ratio': round(out_deg / (in_deg + 1.0), 4),
        'fan_in_ratio': round(in_deg / (out_deg + 1.0), 4),
        'is_burst': 0,
        'is_outlier': 0 # Normal ground truth
    })

# 600 Zero-Day Anomaly Transactions (Sudden high velocity, peeling chains, massive amount outliers)
for i in range(600):
    amt = float(np.random.uniform(30.0, 300.0))
    in_deg = int(np.random.randint(1, 4))
    out_deg = int(np.random.randint(10, 35))
    vel = float(np.random.uniform(1.0, 15.0)) # rapid burst
    records.append({
        'txId': f'anom_{i}',
        'in_degree': in_deg,
        'out_degree': out_deg,
        'tx_amount': round(amt, 6),
        'tx_fee': round(amt * np.random.uniform(0.003, 0.02), 6),
        'hop_velocity': round(vel, 2),
        'fan_out_ratio': round(out_deg / (in_deg + 1.0), 4),
        'fan_in_ratio': round(in_deg / (out_deg + 1.0), 4),
        'is_burst': 1,
        'is_outlier': 1 # Anomalous ground truth
    })

df = pd.DataFrame(records).sample(frac=1.0, random_state=42).reset_index(drop=True)
feature_cols = ['in_degree', 'out_degree', 'tx_amount', 'tx_fee', 'hop_velocity', 'fan_out_ratio', 'fan_in_ratio', 'is_burst']

print(f'Total Dataset: {len(df)} transactions')
df.head()

### 3. Normalize Features with RobustScaler

In [ ]:
scaler = RobustScaler()
X_all = df[feature_cols].values
X_scaled = scaler.fit_transform(X_all)

# Fit Isolation Forest on Normal Baseline to establish benign profile
normal_mask = (df['is_outlier'] == 0)
X_train_normal = X_scaled[normal_mask]
print(f'Training Isolation Forest on {len(X_train_normal)} normal baseline transactions.')

### 4. Train the Isolation Forest Model

In [ ]:
iso_forest = IsolationForest(
    n_estimators=120,
    contamination=0.12,
    max_features=1.0,
    random_state=42
)
iso_forest.fit(X_train_normal)

# Evaluate decision function across full test population
scores = iso_forest.decision_function(X_scaled)
# Convert raw decision score to [0, 1] probability (higher = more anomalous)
anomaly_probs = 1.0 / (1.0 + np.exp(scores * 5.0))
df['anomaly_score'] = anomaly_probs

auc = roc_auc_score(df['is_outlier'], anomaly_probs)
print(f'[*] Isolation Forest ROC-AUC on Zero-Day Anomalies: {auc:.4f}')

### 5. Visualize Anomaly Score Distribution (Normal vs. Outliers)

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(df[df['is_outlier']==0]['anomaly_score'], bins=30, alpha=0.7, color='green', label='Normal Transactions')
plt.hist(df[df['is_outlier']==1]['anomaly_score'], bins=30, alpha=0.7, color='red', label='Zero-Day Outliers')
plt.axvline(0.60, color='black', linestyle='--', label='Threat Threshold (0.60)')
plt.title('Isolation Forest: Anomaly Score Separation')
plt.xlabel('Calculated Anomaly Probability')
plt.ylabel('Transaction Count')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 6. Test on Zero-Day Threat Scenarios (e.g. 2 BTC -> 299 BTC Spike)

In [ ]:
def test_anomaly(name, in_deg, out_deg, amount, vel, fee):
    sample = np.array([[in_deg, out_deg, amount, fee, vel, out_deg/(in_deg+1), in_deg/(out_deg+1), 1 if vel<20 else 0]])
    sample_scaled = scaler.transform(sample)
    raw = iso_forest.decision_function(sample_scaled)[0]
    prob = 1.0 / (1.0 + np.exp(raw * 5.0))
    print(f'[{name}] -> Anomaly Score: {prob*100:.1f}% | Outlier: {"YES (FLAGGED)" if prob > 0.6 else "NO (CLEAN)"}')

test_anomaly('Normal Retail Tx (1.5 BTC, normal delay)', 1, 2, 1.5, 450.0, 0.0001)
test_anomaly('Wallet Drain Burst (299 BTC, 4.2s delay)', 1, 2, 299.0, 4.2, 0.015)
test_anomaly('CoinJoin Peeling Fan-out (25 outputs)', 1, 25, 20.0, 10.0, 0.008)

### 7. Save Model Artifacts

In [ ]:
joblib.dump(iso_forest, 'models/anomaly_detector.joblib')
joblib.dump(scaler, 'models/scaler_anomaly.joblib')
print('[SUCCESS] Isolation Forest Model & Scaler exported to models/anomaly_detector.joblib')